# LSTM for CMAPSS FD001 RUL Prediction (corrected)

## What changed from `04_LSTM_with_all_sensor.ipynb`, and why

Your last run: **R² = 0.636–0.67, train MAE ~20–21 vs val MAE fluctuating
32–38** — a real, widening train/val gap (genuine overfitting), plus a val
loss that bounces around a lot epoch to epoch (2957 → 2058 → 2952 → 3022 →
2209...). Two separate problems were stacked together, so here's what's
fixed and why, in priority order:

1. **Switched input data to the constant-sensor-removed + RUL-capped
   sequences** (18 features instead of 24, RUL capped at 125 instead of
   raw up to 332). This is a data-level fix, not architecture — it removes
   unlearnable target variance and pure-noise input columns. Expect this
   alone to move R² up meaningfully regardless of what the model looks
   like.
2. **Removed `BatchNormalization` from inside the recurrent stack.**
   BatchNorm normalizes over the *batch* dimension at every timestep, and
   its running statistics frequently don't transfer well between training
   and inference in a stacked RNN — this is a known instability source,
   and it lines up with the noisy epoch-to-epoch val_loss you saw
   (2957→2058→2952→3022 is not just healthy noise, it's a sign the
   normalization statistics aren't settling). Swapped for nothing —
   `LayerNormalization` is the safer alternative when normalization is
   needed inside a recurrent stack, but the added capacity here doesn't
   need it once the target/inputs are cleaned up.
3. **Reduced from two stacked `Bidirectional` layers (344K params) to
   one.** The previous model doubled parameters twice on top of an
   already-small dataset (13.8K training sequences) — that's a lot of
   capacity relative to the data, which is exactly the setup that
   overfits. One Bidirectional layer is still a legitimate capacity
   increase (you have the full window at inference time, so there's no
   reason to withhold the backward pass), just not stacked twice.
4. **Kept dropout modest (0.2) and did NOT add L2 on top of it.** Per the
   "don't stack regularizers defensively" rule — add L2 only if this run
   *still* shows a clear overfitting gap; adding it preemptively risks
   pushing back into underfitting the way BatchNorm+L2+Bidirectional all
   at once did last time.

**Test this as one change-set against your baseline**, then if you want to
push further, adjust ONE thing at a time from here (e.g. try adding
LayerNormalization back in, or a second Bidirectional layer) so you can
tell what actually moved the number.

## Realistic expectation

Published FD001 benchmarks (capped RUL, RMSE-based) sit around RMSE 11-14,
which is roughly R² 0.85-0.92. That's the realistic ceiling to aim for —
not 0.96-0.98, which on this dataset usually signals a leakage or
target-mismatch bug rather than a genuinely better model.


In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
RANDOM_SEED = 42

# Setting random seeds for reproducibility
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

WINDOW_SIZE_EXPECTED = 30
NUMBER_OF_FEATURES_EXPECTED = 18  # constant sensors removed (24 - 6)

## Data paths (CHANGED)

Switched from `(not_removed_the_sensor)` to `(removed_the_sensor)` — these
sequences also now carry the capped RUL target once you've re-run
`calculate_rul.ipynb` → `spliting_data.ipynb` → `03_Feature_Scaling.ipynb`
→ `create_sequence.py` in that order.


In [ ]:
X_TRAIN_PATH = (
    "../CMAPSSData/Processed/"
    "X_train_sequences(not_removed_the_sensor).npy"
)

Y_TRAIN_PATH = (
    "../CMAPSSData/Processed/"
    "y_train_sequences(not_removed_the_sensor).npy"
)

X_VAL_PATH = (
    "../CMAPSSData/Processed/"
    "X_val_sequences(not_removed_the_sensor).npy"
)

Y_VAL_PATH = (
    "../CMAPSSData/Processed/"
    "y_val_sequences(not_removed_the_sensor).npy"
)

In [ ]:
BEST_MODEL_PATH = (
    "../Models/Reduced_Sensor_Capped/"
    "LSTM_FD001_reduced_capped_best.keras"
)

FINAL_MODEL_PATH = (
    "../Models/Reduced_Sensor_Capped/"
    "LSTM_FD001_reduced_capped_final.keras"
)

HISTORY_PATH = (
    "../Models/Reduced_Sensor_Capped/"
    "LSTM_FD001_reduced_capped_history.npz"
)

In [ ]:
X_train = np.load(X_TRAIN_PATH)
y_train = np.load(Y_TRAIN_PATH)

X_val = np.load(X_VAL_PATH)
y_val = np.load(Y_VAL_PATH)


print("\nTraining data:")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nValidation data:")
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

In [ ]:
print("\nX_train dtype:", X_train.dtype)
print("y_train dtype:", y_train.dtype)

print("\nX_train minimum:", X_train.min())
print("X_train maximum:", X_train.max())

print("\ny_train minimum:", y_train.min())
print("y_train maximum:", y_train.max())

assert y_train.max() <= 125, (
    "y_train max exceeds 125 -- RUL capping was not applied "
    "upstream. Re-run calculate_rul.ipynb through create_sequence.py."
)

In [ ]:
print("DATA VALIDATION")

if X_train.ndim != 3:
    raise ValueError(
        f"X_train must be 3-dimensional. "
        f"Found shape: {X_train.shape}"
    )

if X_val.ndim != 3:
    raise ValueError(
        f"X_val must be 3-dimensional. "
        f"Found shape: {X_val.shape}"
    )

window_size = X_train.shape[1]

if window_size != WINDOW_SIZE_EXPECTED:
    print(
        f"\nWARNING: Expected window size "
        f"{WINDOW_SIZE_EXPECTED}, "
        f"but found {window_size}."
    )

number_of_features = X_train.shape[2]

if number_of_features != NUMBER_OF_FEATURES_EXPECTED:
    print(
        f"\nWARNING: Expected {NUMBER_OF_FEATURES_EXPECTED} features "
        f"(constant sensors removed), but found {number_of_features}. "
        f"Check you loaded the (removed_the_sensor) files."
    )

if X_val.shape[2] != number_of_features:
    raise ValueError(
        "Training and validation datasets have "
        "different numbers of features."
    )

if len(X_train) != len(y_train):
    raise ValueError(
        "X_train and y_train have different "
        "numbers of samples."
    )

if len(X_val) != len(y_val):
    raise ValueError(
        "X_val and y_val have different "
        "numbers of samples."
    )

print("\nNaN values:")

print("X_train:", np.isnan(X_train).sum())
print("y_train:", np.isnan(y_train).sum())
print("X_val:", np.isnan(X_val).sum())
print("y_val:", np.isnan(y_val).sum())


print("\nInfinite values:")

print("X_train:", np.isinf(X_train).sum())
print("y_train:", np.isinf(y_train).sum())
print("X_val:", np.isinf(X_val).sum())
print("y_val:", np.isinf(y_val).sum())


if (
    np.isnan(X_train).any()
    or np.isnan(y_train).any()
    or np.isnan(X_val).any()
    or np.isnan(y_val).any()
):
    raise ValueError("NaN values detected in the dataset.")

if (
    np.isinf(X_train).any()
    or np.isinf(y_train).any()
    or np.isinf(X_val).any()
    or np.isinf(y_val).any()
):
    raise ValueError(
        "Infinite values detected in the dataset."
    )

In [ ]:
print("INPUT INFORMATION")

print("\nWindow size:", window_size)
print("Number of features:", number_of_features)
print(
    "Input shape:",
    X_train.shape[1:]
)

print("\nTraining samples:", len(X_train))
print("Validation samples:", len(X_val))


print("\nTraining RUL range:")
print("Minimum:", y_train.min())
print("Maximum:", y_train.max())

print("\nValidation RUL range:")
print("Minimum:", y_val.min())
print("Maximum:", y_val.max())

## Model architecture (CHANGED)

Single `Bidirectional(LSTM(64))` (not two stacked), no `BatchNormalization`,
moderate `Dropout(0.2)` between layers, no L2. ~half the parameter count of
the previous 344K-param version, sized more sensibly for 13.8K training
sequences.


In [ ]:
print("Building LSTM Model")

INPUT_SHAPE = X_train.shape[1:]

model = Sequential([
    Input(shape=INPUT_SHAPE),

    Bidirectional(
        LSTM(64, return_sequences=True)
    ),
    Dropout(0.2),

    LSTM(32, return_sequences=False),
    Dropout(0.2),

    Dense(16, activation="relu"),
    Dense(1)
])

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

In [ ]:
print("LSTM Model Architecture")
model.summary()

In [ ]:
print("SETTING TRAINING CALLBACKS")

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=12,
    mode='min',
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-5,
    mode='min',
    verbose=1
)

model_checkpoint = ModelCheckpoint(
    BEST_MODEL_PATH,
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    verbose=1
)

In [ ]:
print("STARTING LSTM TRAINING")

history = model.fit(
    X_train, y_train,
    validation_data=(
        X_val,
        y_val
    ),
    epochs=100,
    batch_size=64,
    callbacks=[
        early_stopping,
        reduce_lr,
        model_checkpoint
    ],
    verbose=1
)

In [ ]:
model.save(FINAL_MODEL_PATH)

print("MODEL TRAINING COMPLETED")

print("\nFinal model saved at:")
print(FINAL_MODEL_PATH)

print("\nBest model saved at:")
print(BEST_MODEL_PATH)

In [ ]:
np.savez(
    HISTORY_PATH,

    loss=np.array(
        history.history["loss"]
    ),

    val_loss=np.array(
        history.history["val_loss"]
    ),

    mae=np.array(
        history.history["mae"]
    ),

    val_mae=np.array(
        history.history["val_mae"]
    )
)

In [ ]:
print("TRAINING INFORMATION")

print("Total epochs requested", 100)
print("Actual epochs completed:", len(history.history['loss']))
print("Best validation loss:", min(history.history["val_loss"]))
print("Best validation MAE:", min(history.history["val_mae"]))

# Overfitting check: compare final train vs val MAE directly
final_train_mae = history.history['mae'][-1]
final_val_mae = history.history['val_mae'][-1]
gap = final_val_mae - final_train_mae
print(f"\nFinal train MAE: {final_train_mae:.3f}")
print(f"Final val MAE:   {final_val_mae:.3f}")
print(f"Train/val gap:   {gap:.3f} cycles (rule of thumb: >5 cycles gap on this scale suggests real overfitting)")

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(history.history["loss"], label="Training Loss")

plt.plot(history.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training vs Validation Loss")

plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(history.history["mae"], label="Training MAE")

plt.plot(history.history["val_mae"], label="Validation MAE")

plt.xlabel("Epoch")
plt.ylabel("MAE (Cycles)")
plt.title("Training vs Validation MAE")

plt.legend()
plt.grid(True)

plt.tight_layout()

plt.show()

In [ ]:
print("VALIDATION PREDICTION")

y_pred = model.predict(
    X_val,
    verbose=1
)

y_pred = y_pred.reshape(-1)

mae = mean_absolute_error(y_val, y_pred)

mse = mean_squared_error(y_val, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_val, y_pred)

In [ ]:
print("VALIDATION PERFORMANCE")

print(f"\nMAE : {mae:.4f} cycles")

print(f"RMSE  : {rmse:.4f} cycles")

print(f"R²    : {r2:.4f}")

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(y_val, y_pred, alpha=0.5)

# Perfect prediction line

minimum = min(y_val.min(), y_pred.min())

maximum = max(y_val.max(), y_pred.max())

plt.plot([minimum, maximum], [minimum, maximum], linestyle="--")

plt.xlabel("Actual RUL")
plt.ylabel("Predicted RUL")

plt.title("Actual RUL vs Predicted RUL")

plt.grid(True)

plt.tight_layout()

plt.show()

In [ ]:
number_to_plot = min(300, len(y_val))

plt.figure(figsize=(12, 6))

plt.plot(y_val[:number_to_plot], label="Actual RUL")

plt.plot(y_pred[:number_to_plot], label="Predicted RUL")

plt.xlabel("Validation Sample")
plt.ylabel("RUL (Cycles)")

plt.title("Actual vs Predicted RUL on Validation Data")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
print("FINAL SUMMARY")

print("""
Experiment:
    Constant sensors removed, RUL capped at 125

Model:
    Bidirectional(LSTM(64))
    Dropout(0.20)
    LSTM(32)
    Dropout(0.20)
    Dense(16, ReLU)
    Dense(1)

Optimizer:
    Adam

Initial learning rate:
    0.001

Loss:
    MSE

Training metric:
    MAE

Maximum epochs:
    100

Batch size:
    64

Callbacks:
    EarlyStopping
    ReduceLROnPlateau
    ModelCheckpoint
""")

print(f"Validation MAE  : {mae:.4f} cycles")

print(f"Validation RMSE : {rmse:.4f} cycles")

print(f"Validation R²   : {r2:.4f}")

print("\nTraining completed successfully.")